# Libraries

In [1]:
# initialization
# python -m venv .env
# .env\Scripts\activate

# package installations
#%pip install mlflow 
#%pip install openai 
#%pip install python-dotenv
#%pip install pandas

In [2]:
import mlflow
import requests
import json
import time
import sys
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime

# Variables

In [3]:
# --- Configuration ---
# The URL of your local Next.js Chat API
API_URL = "http://localhost:3000/api/chat"

# The URL of your MLflow server (from docker-compose)
MLFLOW_TRACKING_URI = "http://localhost:5000"

# Test Credentials (mock values - update with real IDs if your DB requires them)
USER_ID = "mlflow-test-user"
SITE_ID = "mlflow-test-site"

# Load environment variables from the Next.js app folder
env_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'xm-cloud-chatbot', '.env'))
if os.path.exists(env_path):
    print(f"Loading .env from {env_path}")
    load_dotenv(env_path)
else:
    print("Warning: .env file not found. Ensure OPENAI_API_KEY is set.")

Loading .env from c:\src\MultimodalMarketingDemo\xm-cloud-chatbot\.env


# Dataset

In [4]:
# The questions to test the chatbot with
TEST_DATASET = [
    "Can you list the available sites?",
    "Can you find the multimodalmarketing site?",
    "What pages are in the multimodalmarketing site?",
    "What components are on the healthy diet page of the multimodalmarketing site?",
]

# LLM as a Judge

In [5]:
# Initialize OpenAI client for the "Judge"
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Functions

In [6]:
def evaluate_response(question, answer):
    """
    Uses OpenAI to evaluate the quality of the chatbot's response.
    Returns a dict with 'score' and 'reasoning'.
    """
    if not answer or "Error:" in answer:
        return {"score": 0, "reasoning": "Failed to get a valid response from the chatbot."}

    prompt = f"""
    You are an expert judge evaluating the performance of a Sitecore XM Cloud specialized chatbot.
    
    User Question: {question}
    Chatbot Answer: {answer}

    Please evaluate the answer on a scale of 1 to 5 (5 being best) based on:
    1. Relevance: Did it answer the user's question directly?
    2. Helpfulness: key Sitecore specific information provided?
    3. Clarity: Is the answer easy to understand?

    Return ONLY valid JSON in the following format:
    {{
      "score": <number>,
      "reasoning": "<short explanation>"
    }}
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-4", # Or gpt-3.5-turbo if cost is a concern
            messages=[
                {"role": "system", "content": "You are a helpful assistant that evaluates chatbot interactions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        content = response.choices[0].message.content
        # Clean up code blocks if the model adds them
        if content.startswith("```json"):
            content = content[7:-3]
        elif content.startswith("```"):
            content = content[3:-3]
            
        return json.loads(content)
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return {"score": 0, "reasoning": f"Evaluation failed: {e}"}

In [ ]:
def parse_sse_stream(response):
    """
    Parses a text/event-stream response to extract the actual message content.
    Assumes "data: <json>" format where json contains { "type": "content", "content": "..." }
    """
    full_text = ""
    for line in response.iter_lines():
        if line:
            line = line.decode('utf-8')
            if line.startswith('data: '):
                content = line[6:] # Strip 'data: '
                if content.strip() == "[DONE]":
                    break
                try:
                    # Parse the JSON event from the stream
                    data = json.loads(content)
                    
                    # Next.js Chat API sends events like { type: 'content', content: '...' }
                    if isinstance(data, dict):
                        event_type = data.get("type")
                        if event_type == "content":
                            full_text += data.get("content", "")
                        elif event_type == "error":
                            full_text += f"\n[Error: {data.get('message')}]"
                        # Ignore other types (title, usage, done, etc.) for the main output
                    else:
                        # Fallback if structure is unknown
                        full_text += str(data)
                except json.JSONDecodeError:
                    # If it's not JSON, append raw text
                    full_text += content
    return full_text

In [8]:
def run_evaluation(evaluation_name, experiment_name):
    print(f"Connecting to MLflow at {MLFLOW_TRACKING_URI}...")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=evaluation_name) as run:
        print(f"Active Run ID: {run.info.run_id}")
        print(f"Artifact URI: {run.info.artifact_uri}") # Debug: Print where it thinks artifacts go
        
        # Lists to store results for the table
        results = []

        for question in TEST_DATASET:
            print(f"Asking: '{question}'...")
            start_time = time.time()
            
            payload = {
                "message": question,
                "userId": USER_ID,
                "siteId": SITE_ID,
                # Optional context to mimic the real app
                "applicationId": "mlflow-test-app"
            }

            try:
                # Next.js app running locally
                response = requests.post(API_URL, json=payload, stream=True)
                
                latency = time.time() - start_time
                status_code = response.status_code
                
                response_text = ""
                if status_code == 200:
                    # Consume the stream
                    response_text = parse_sse_stream(response)
                else:
                    response_text = f"Error: {response.text}"

                print(f" -> Status: {status_code}, Latency: {latency:.2f}s")
                
                # Perform LLM-as-a-Judge Evaluation
                print(" -> Evaluating response quality...")
                eval_result = evaluate_response(question, response_text)
                print(f" -> Score: {eval_result['score']}/5")

                # Store result
                results.append({
                    "question": question,
                    "answer": response_text,
                    "latency": latency,
                    "status_code": status_code,
                    "score": eval_result.get("score", 0),
                    "reasoning": eval_result.get("reasoning", "N/A")
                })

            except Exception as e:
                print(f" -> Exception: {e}")
                results.append({
                    "question": question,
                    "answer": str(e),
                    "latency": time.time() - start_time,
                    "status_code": 0,
                    "score": 0,
                    "reasoning": "Exception during execution"
                })

        # Convert to DataFrame for easier logging and analysis
        df = pd.DataFrame(results)

        # Log metrics
        avg_latency = df["latency"].mean()
        avg_score = df["score"].mean()
        
        mlflow.log_metric("average_latency", avg_latency)
        mlflow.log_metric("average_quality_score", avg_score)
        
        try:
            # Log the dataset as a table artifact
            print("Logging table artifact...")
            mlflow.log_table(data=df, artifact_file="test_results.json")
            print("Table artifact logged successfully.")
        except Exception as e:
            print(f"FAILED to log table artifact: {e}")

        print("\nEvaluation Complete.")
        print(f"View results at: {MLFLOW_TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")
        return df

# Evaluation

In [9]:
if __name__ == "__main__":
    
    experiment_name = "Chatbot_Response_Quality_v2" 
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"smoke_test_v1_{timestamp}"

    run_evaluation(run_name, experiment_name)

2026/01/28 10:21:29 INFO mlflow.tracking.fluent: Experiment with name 'Chatbot_Response_Quality_v2' does not exist. Creating a new experiment.


Connecting to MLflow at http://localhost:5000...
Active Run ID: e0b66b84584a46edbb2e1d583f0c9976
Artifact URI: mlflow-artifacts:/2/e0b66b84584a46edbb2e1d583f0c9976/artifacts
Asking: 'Can you list the available sites?'...
 -> Status: 200, Latency: 2.58s
 -> Evaluating response quality...
 -> Score: 2/5
Asking: 'Can you find the multimodalmarketing site?'...
 -> Status: 200, Latency: 1.89s
 -> Evaluating response quality...
 -> Score: 2/5
Asking: 'What pages are in the multimodalmarketing site?'...
 -> Status: 200, Latency: 2.81s
 -> Evaluating response quality...
 -> Score: 2/5
Asking: 'What components are on the healthy diet page of the multimodalmarketing site?'...
 -> Status: 200, Latency: 2.12s
 -> Evaluating response quality...
 -> Score: 2/5
Logging table artifact...
Table artifact logged successfully.

Evaluation Complete.
View results at: http://localhost:5000/#/experiments/2/runs/e0b66b84584a46edbb2e1d583f0c9976
🏃 View run smoke_test_v1_20260128_102129 at: http://localhost:5000